# Fundamentos de RAG

RAG (Retrieval-Augmented Generation) es una técnica que combina la recuperación de información con modelos de lenguaje generativos para producir respuestas más precisas y actualizadas. En lugar de depender únicamente del conocimiento interno de un modelo de IA, RAG primero busca información relevante en una base de datos o documentos externos, y luego utiliza esa información como contexto para generar una respuesta fundamentada. 

Esta arquitectura resuelve problemas comunes de los LLMs como las alucinaciones (inventar información), el conocimiento desactualizado y la falta de fuentes verificables. 

RAG se ha convertido en un componente esencial para construir asistentes de IA empresariales, chatbots especializados y sistemas de pregunta-respuesta que requieren acceso a información específica de dominio.

In [22]:
# Dataset con 50 datos sobre Argentina

documents = [
    "Argentina es el segundo país más grande de América del Sur y el octavo más grande del mundo.",
    "Buenos Aires es la capital y ciudad más grande de Argentina.",
    "El tango es un baile y género musical originario de Argentina, especialmente de Buenos Aires.",
    "Argentina limita con Chile, Bolivia, Paraguay, Brasil y Uruguay.",
    "El Aconcagua, ubicado en Argentina, es la montaña más alta de América con 6,961 metros.",
    "Argentina es famosa por su carne de res de alta calidad y sus asados tradicionales.",
    "El idioma oficial de Argentina es el español, con un acento distintivo rioplatense.",
    "Las Cataratas del Iguazú, compartidas con Brasil, son una de las maravillas naturales del mundo.",
    "Argentina ganó la Copa Mundial de la FIFA en 1978, 1986 y 2022.",
    "El peso argentino es la moneda oficial del país.",
    "Diego Maradona y Lionel Messi son dos de los futbolistas más famosos de Argentina.",
    "La Pampa argentina es una extensa llanura conocida por su producción agrícola y ganadera.",
    "El glaciar Perito Moreno es uno de los glaciares más famosos y accesibles del mundo.",
    "Argentina declaró su independencia de España el 9 de julio de 1816.",
    "El mate es la bebida tradicional argentina, consumida en todo el país.",
    "La Patagonia argentina es conocida por sus paisajes impresionantes y vida silvestre única.",
    "Argentina es uno de los principales productores y exportadores de vino del mundo.",
    "La región vinícola de Mendoza es famosa por su producción de vino Malbec.",
    "El Teatro Colón en Buenos Aires es uno de los teatros de ópera más importantes del mundo.",
    "Argentina tiene una población de aproximadamente 45 millones de habitantes.",
    "La Avenida 9 de Julio en Buenos Aires es una de las avenidas más anchas del mundo.",
    "El fútbol es el deporte más popular en Argentina.",
    "Argentina tiene cinco sitios declarados Patrimonio de la Humanidad por la UNESCO.",
    "La región de Cuyo incluye las provincias de Mendoza, San Juan y San Luis.",
    "Argentina es el país hispanohablante más austral del mundo.",
    "El Obelisco de Buenos Aires es un icónico monumento nacional inaugurado en 1936.",
    "La Quebrada de Humahuaca en Jujuy es Patrimonio de la Humanidad por la UNESCO.",
    "Argentina tiene una importante comunidad de descendientes de italianos, españoles y alemanes.",
    "El dulce de leche es un producto tradicional argentino utilizado en numerosos postres.",
    "La Tierra del Fuego es la provincia más austral de Argentina.",
    "El Parque Nacional Los Glaciares es uno de los parques más visitados del país.",
    "Argentina es un importante productor de soja, trigo y maíz.",
    "La bandera argentina tiene tres franjas horizontales: celeste, blanca y celeste, con un sol en el centro.",
    "Jorge Luis Borges es uno de los escritores argentinos más reconocidos internacionalmente.",
    "El tren a las nubes en Salta alcanza alturas de más de 4,200 metros sobre el nivel del mar.",
    "Argentina tiene costas sobre el Océano Atlántico de más de 4,700 kilómetros.",
    "La Casa Rosada es la sede del gobierno argentino y un emblema de Buenos Aires.",
    "Ushuaia es considerada la ciudad más austral del mundo.",
    "Argentina tiene 23 provincias y una ciudad autónoma (Buenos Aires).",
    "El Parque Nacional Iguazú protege las famosas Cataratas del Iguazú.",
    "La economía argentina es la tercera más grande de América Latina.",
    "El empanada es una comida tradicional argentina que se consume en todo el país.",
    "Argentina tiene una rica tradición literaria con autores como Julio Cortázar y Adolfo Bioy Casares.",
    "El Cerro Catedral en Bariloche es uno de los centros de esquí más importantes de Sudamérica.",
    "La Península Valdés es un importante santuario de vida marina, especialmente de ballenas.",
    "Argentina es miembro fundador de las Naciones Unidas y del Mercosur.",
    "El clima de Argentina varía desde subtropical en el norte hasta subpolar en el sur.",
    "La Universidad de Buenos Aires (UBA) es una de las universidades más prestigiosas de América Latina.",
    "El chimichurri es una salsa argentina típica que acompaña las carnes asadas.",
    "Argentina tiene una destacada escena de rock nacional con bandas como Soda Stereo y Los Fabulosos Cadillacs."
]

In [ ]:
# en caso de no tener librerias instalarlas!

# !pip install langchain-huggingface sentence-transformers langchain_community faiss-cpu -q

# para instalar faiss en windows 
# # si usas linux o mac usa: !pip install faiss-gpu   
# # si usas mac y da error instala con: !pip install faiss-cpu    


In [24]:
# importnado las librerias necesarias

import os # para manejar variables de entorno
from langchain.docstore.document import Document # para manejar documentos
from langchain.vectorstores.faiss import FAISS # para manejar la base de datos vectorial
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint # para manejar modelos de huggingface
from langchain_core.messages import HumanMessage, SystemMessage # para manejar mensajes de chat
from IPython.display import display, Markdown # para mostrar mensajes en jupyter

In [ ]:
# configurando la key de huggingface 

# get a token: https://huggingface.co/docs/api-inference/quicktour#get-your-api-token <- GENERA TU TOKEN AQUI!!!

from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass() # para agregar el token de manera segura
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

## 1- Creando los embeddings (representacion matematica de nuestros datos)

Los **embeddings** son la forma en que convertimos nuestros textos en **vectores numéricos** que una IA puede entender y comparar.  
En otras palabras, transforman frases o documentos en listas de números que capturan **significado y contexto**, no solo palabras sueltas.

---

### 📌 ¿Por qué necesitamos embeddings?

Los modelos de IA no “entienden” el lenguaje directamente. Necesitan trabajar con números.  
Los embeddings permiten representar ideas similares con vectores similares, lo que es clave para tareas como:

- Búsqueda semántica 🔍  
- Agrupamiento de documentos 🗂️  
- Preguntas y respuestas sobre un corpus ❓  
- Recomendaciones inteligentes 🤝

**Ejemplo:**

| Texto | Embedding (simplificado) |
|-------|----------------------------|
| “Buenos Aires es la capital de Argentina.” | [0.12, -0.45, 0.87, …] |
| “La capital argentina es Buenos Aires.”     | [0.11, -0.47, 0.86, …] |

Aunque las frases son diferentes, sus embeddings son **muy parecidos**, porque expresan la misma idea.

---

### 🧭 Proceso general

1. **Preparamos los datos**  
   Por ejemplo, cargamos nuestras frases desde un archivo `documents.json`.

2. **Enviamos cada texto al modelo de embeddings**  
   El modelo convierte cada texto en un vector de alta dimensión (por ejemplo, 1 536 números).

3. **Guardamos estos vectores**  
   Podemos almacenarlos en memoria, en una base de datos vectorial, o en un índice para búsquedas posteriores.

---

📊 Resultado

*   Cada documento ahora tiene su vector matemático, que se puede:

*   Comparar usando métricas como cosine similarity para encontrar textos parecidos.

*   Indexar en herramientas como FAISS, Pinecone o Chroma para búsquedas rápidas.

*   Reutilizar en tareas de RAG (Retrieval-Augmented Generation).


In [26]:
docs = [Document(page_content=text) for text in documents] # creando una lista de documentos que luego usaremos para crear los embeddings
docs[:5] # mostrando los primeros 5 documentos

[Document(metadata={}, page_content='Argentina es el segundo país más grande de América del Sur y el octavo más grande del mundo.'),
 Document(metadata={}, page_content='Buenos Aires es la capital y ciudad más grande de Argentina.'),
 Document(metadata={}, page_content='El tango es un baile y género musical originario de Argentina, especialmente de Buenos Aires.'),
 Document(metadata={}, page_content='Argentina limita con Chile, Bolivia, Paraguay, Brasil y Uruguay.'),
 Document(metadata={}, page_content='El Aconcagua, ubicado en Argentina, es la montaña más alta de América con 6,961 metros.')]

In [27]:
# usa un modelo transformer de huggingface para crear los embeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # modelo pequeño y rapido

# pero hay muchos otros modelos disponibles en huggingface en la pagina: https://huggingface.co/models?pipeline_tag=feature-extraction&sort=downloads


## 2- Creando un FAISS Vectorstore con los documentos y el modelo de embeddings

Una vez que generamos los **embeddings**, necesitamos una forma eficiente de **almacenarlos** y **consultarlos**.  
Ahí es donde entra **FAISS** (*Facebook AI Similarity Search*), una librería optimizada para realizar búsquedas vectoriales de manera rápida y escalable 🧠⚡.

---

## 📌 ¿Qué es FAISS?

**FAISS** es una biblioteca desarrollada por Facebook AI Research que permite:

- Guardar grandes cantidades de vectores (embeddings).  
- Realizar búsquedas por similitud de forma muy eficiente.  
- Escalar fácilmente con millones de documentos.  

En lugar de recorrer todos los embeddings uno por uno, FAISS utiliza estructuras internas optimizadas para encontrar rápidamente los vectores más parecidos a una consulta.

---

In [ ]:
# creando un FAISS vectorstore con los documentos y el modelo de embeddings
# Un faiss vectorstore es una base de datos que almacena vectores y permite realizar búsquedas rápidas de similitud entre ellos.

faiss_store = FAISS.from_documents(docs, embedding_model) # creando el vectorstore
faiss_store.index # mostrando el indice del vectorstore

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001CA40323D20> >

In [29]:
# Mostrando el numero de vectores en el indice

print(f"Numero total de indices: {faiss_store.index.ntotal}") # mostrando el numero total de vectores en el indice

Numero total de indices: 50


In [30]:
# Mostrando la cantidad de vectores o dimensiones de cada indice

print(f"Dimensiones de cada vector: {faiss_store.index.d}") # mostrando las dimensiones de cada vector

# esto significa que cada vector tiene 384 dimensiones, es decir, cada documento es representado por un vector en un espacio de 384 dimensiones.
# estas dimensiones son las que permiten comparar la similitud entre documentos y consultas.

Dimensiones de cada vector: 384


In [31]:
# Mostrando los embeddings del primer documento o vector   
 
print(f"Embeddings del primer vector: {faiss_store.index.reconstruct(0)}") # mostrando los embeddings del primer vector

# aca estamos viendo los embeddings del primer documento, que es una representacion matematica del texto original.
# estos valores son los que se usan para comparar la similitud entre documentos y consultas.

Embeddings del primer vector: [ 2.84775738e-02 -2.60242308e-03 -3.59398909e-02 -3.12094577e-02
 -6.98101195e-03  1.48279164e-02 -1.24294991e-02 -4.74726446e-02
  2.66768150e-02  5.92028238e-02 -6.58167200e-03 -1.43645601e-02
 -7.61042237e-02 -5.16469683e-03  3.48336622e-02 -1.40086310e-02
 -3.85704637e-02  8.26807413e-03  2.52461415e-02 -1.76379271e-02
  2.16704190e-01 -2.36915592e-02 -4.29612324e-02  3.15154977e-02
 -2.00168416e-02 -5.36655658e-04 -1.71099498e-03 -3.56190950e-02
 -8.02086946e-03 -2.42037904e-02  2.69534104e-02  3.92186828e-02
  7.98718780e-02  1.08497804e-02 -7.89463520e-03  1.79936364e-02
  7.18332874e-03 -7.23057166e-02  1.90101136e-02  2.18945816e-02
 -7.55588561e-02  2.88483594e-03  4.01838012e-02 -1.90155208e-02
  6.39067777e-03 -3.98558453e-02  2.30535660e-02  1.17523007e-01
  1.47440340e-02  1.64004171e-03  6.90473244e-02 -7.87913054e-02
 -1.54276639e-01  2.76112533e-03  2.13853922e-02  2.07453221e-02
  4.60473187e-02 -9.06725898e-02  8.62552822e-02  8.89378712

## 3- Recuperando documentos similares a una consulta

Una vez que tenemos nuestro **FAISS Vectorstore** creado y listo, el siguiente paso es **recuperar los documentos más similares a una consulta**.  
Este proceso es fundamental en sistemas de **búsqueda semántica** y **RAG (Retrieval-Augmented Generation)**, ya que permite encontrar la información más relevante **basándose en significado**, no en coincidencias exactas de palabras.

---

## 📌 ¿Qué vamos a hacer?

1. **Definir una consulta**: Es el texto o pregunta que el usuario realiza.  
2. **Convertir la consulta en un embedding**: Internamente, FAISS generará un vector para la consulta usando el mismo modelo de embeddings que se usó para los documentos.  
3. **Buscar en el vectorstore**: Se compara el embedding de la consulta con todos los vectores almacenados, y FAISS devuelve los documentos **más cercanos en el espacio vectorial**.  
4. **Obtener los scores de similitud**: Cada documento devuelto incluye un valor numérico que indica **qué tan cerca está del embedding de la consulta**.  
   - Cuanto **más bajo** sea el score, **mayor similitud**.

---


In [ ]:
# recuperando documentos similares a una consulta

# consulta de ejemplo
query = "¿En que continente esta Argentina?" 
k = 3 # numero de documentos a recuperar es decir 3 documentos mas similares a la consulta

# consultando el vectorstore para recuperar los documentos mas similares a la consulta con su score de similitud
similar_docs_score = faiss_store.similarity_search_with_score(query, k=k)
similar_docs_score

# el score mas bajo es el mas similar, eso significa que el primer documento es el mas relevante para la consulta
# un valor bajo de score significa que esta mas cerca en el espacio vectorial al vector de la consulta

[(Document(id='e9dfaacd-1e1e-4b82-a71d-2039145780a7', metadata={}, page_content='Argentina es el segundo país más grande de América del Sur y el octavo más grande del mundo.'),
  np.float32(0.5963669)),
 (Document(id='46a91e51-6f42-4135-a9ce-dddbab2d28f5', metadata={}, page_content='La economía argentina es la tercera más grande de América Latina.'),
  np.float32(0.61913717)),
 (Document(id='7d80a078-ebc6-4e03-a87d-7786a11c6de9', metadata={}, page_content='Argentina tiene una importante comunidad de descendientes de italianos, españoles y alemanes.'),
  np.float32(0.6257247))]

In [ ]:
# podemos simplemente ver cuales son los documentos mas similares a la consulta

retrieved_docs = faiss_store.similarity_search(query, k=k)
retrieved_docs

[Document(id='e9dfaacd-1e1e-4b82-a71d-2039145780a7', metadata={}, page_content='Argentina es el segundo país más grande de América del Sur y el octavo más grande del mundo.'),
 Document(id='46a91e51-6f42-4135-a9ce-dddbab2d28f5', metadata={}, page_content='La economía argentina es la tercera más grande de América Latina.'),
 Document(id='7d80a078-ebc6-4e03-a87d-7786a11c6de9', metadata={}, page_content='Argentina tiene una importante comunidad de descendientes de italianos, españoles y alemanes.')]

In [ ]:
# creando una funcion para mostrar los resultados de manera mas amigable
def show_results(query,k):
    return faiss_store.similarity_search(query, k=k)

## 4- Desarrollando el modelo generativo

Después de recuperar los documentos más relevantes desde el vectorstore, el siguiente paso es **usar un modelo de lenguaje (LLM)** para **generar respuestas en lenguaje natural** basadas en esa información.  
En este caso, vamos a **cargar un modelo desde Hugging Face** para realizar tareas de **generación de texto** 🧠✍️.

---

## 📌 ¿Qué vamos a hacer?

1. **Seleccionar un modelo de Hugging Face**  
   Se elige un modelo preentrenado especializado en generación de texto.  
   En este ejemplo usamos:  
   - 🧠 `mistralai/Mixtral-8x7B-Instruct-v0.1` → un modelo grande, potente y afinado para instrucciones.

2. **Configurar la tarea de generación de texto**  
   Indicamos que el modelo será utilizado para *text generation*, es decir, producir respuestas coherentes y contextuales a partir de un *prompt*.

3. **Crear el objeto de chat**  
   Transformamos el modelo en una interfaz de chat (`ChatHuggingFace`) para integrarlo fácilmente en flujos de RAG, donde el modelo recibe contexto + pregunta y genera una respuesta.

---
🧠 ¿Por qué usar un LLM aquí?

*   Una vez que encontramos los documentos más relevantes, el modelo de lenguaje se encarga de:

*   Leer el contexto recuperado.

*   Comprender la pregunta del usuario.

*   Generar una respuesta natural, coherente y útil, combinando ambos elementos.

Este es el corazón de la técnica RAG (Retrieval-Augmented Generation):

1-  Primero recuperamos información.

2-  Luego generamos una respuesta basada en esa información.


In [35]:
# cargando un llm de huggingface para generar respuestas

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mixtral-8x7B-Instruct-v0.1", # modelo grande y potente
    task="text-generation" # tarea de generacion de texto
)

chat_model = ChatHuggingFace(llm=llm) # creando el modelo de chat

In [36]:
# definiendo el menaje de sistema para el modelo de chat y el mensaje del usuario.

messages = [SystemMessage(content="Eres un asistente util que ayuda a responder preguntas sobre Argentina y hablas como Argentino."),
            HumanMessage(content="¿En que continente esta Argentina?")
]

In [37]:
# generando la respuesta del modelo de chat

response = chat_model.invoke(messages) #
display(Markdown(f"**Respuesta del modelo:** {response.content}"))

**Respuesta del modelo:**  Argentina se encuentra en el continente de América del Sur. Es el segundo país más grande de Sudamérica y el octavo más grande del mundo en términos de superficie terrestre. Limita al oeste con Chile, al norte y este con Bolivia, Paraguay, Brasil y Uruguay, y al este y sur con el Océano Atlántico. Argentina también comparte un breve límite marítimo con el territorio británico de ultramar de las Islas Malvinas, que es reclamado por Argentina como las Islas Malvinas.

Además de su tamaño y ubicación geográfica, Argentina es conocida por su rica cultura y diversidad, su economía próspera, sus paisajes impresionantes y sus contribuciones a la música, el deporte, la literatura y las artes visuales. La capital de Argentina, Buenos Aires, es una ciudad cosmopolita y vibrante que atrae a visitantes de todo el mundo.

Pero en este punto, la respuesta esta solo dada por el modelo Y NO TIENE EN CUENTA NUESTROS DATOS, POR LO QUE VAMOS A MODIFICAR ESO PARA QUE SE BASE EN NUESTROS DATOS Y GENERE UNA RESPUESTA COMBINADA ENTRE LLM Y NUESTA INFORMACION.

In [38]:
# Generando una respuesta combinada entre nuestros documentos y el modelo de chat
# Creando una funcion que combine la recuperacion de documentos y la generacion de respuestas

# paso 1 creando la funcion para generar la respuesta con llm

def generative_response(query,context):
    messages = [
        SystemMessage(content=f"""Eres un guia turistico Argentino muy gracioso.
                      solo responde teniendo en cuenta {context}.
                      Si no sabes la respuesta di 'No lo se'
                      """),
        HumanMessage(content=f"responde a la {query} basado en los siguientes documentos: {context}")
    ]
    ai_response = chat_model.invoke(messages) # generando la respuesta del modelo de chat
    return ai_response.content

In [ ]:
# combinando las respuestas de llm con los documentos recuperados

def rag(query):
    context = show_results(query,k) # recuperando los documentos mas similares a la consulta
    respuesta = generative_response(query,context) # generando la respuesta con el contexto de los documentos recuperados
    return display(Markdown(respuesta))

In [47]:
# testeando la funcion rag
query = "¿Cual es la capital de Argentina?"
rag(query)

 ¡Hola, amigos! Preguntaron cuál es la capital de Argentina, y ¡buenísimas noticias! Tenemos la respuesta gracias al primer documento ¡en mis manos! ¡Sí, señoras y señores! La capital de Argentina es la ciudad maravillosa y llena de vida conocida como Buenos Aires. ¡Así que ya lo saben, mis queridos turistas! Cuando visiten Argentina, asegúrense de pasar por esta vibrante y espectacular ciudad. ¡No se arrepentirán!

## 5- Testeando con información que no existe en nuestros documentos

Una parte importante del desarrollo de un sistema basado en **búsqueda semántica + LLM (RAG)** es entender **sus límites**.  
Cuando realizamos una consulta sobre un tema que **no está presente en nuestros documentos**, el sistema **no puede “inventar” información real que no exista en la base** (a menos que el modelo generativo alucine).

---

### 📌 ¿Qué ocurre en este caso?

1. **El vectorstore intenta encontrar documentos similares**, pero al no haber coincidencias semánticas relevantes, devolverá documentos con scores **menos precisos** o incluso vacíos.  
2. **El modelo generativo no recibirá contexto útil**, por lo que:
   - Podría **responder que no tiene información suficiente** (si está bien configurado).  
   - O podría **inventar una respuesta incorrecta** (*alucinación*) si no se establecen límites adecuados.

---

### 🧠 Importancia de esta prueba

Este tipo de test nos permite:

- Verificar si nuestro sistema **maneja correctamente las consultas “fuera de dominio”**.  
- Ajustar el comportamiento del LLM para que responda de forma responsable (por ejemplo, diciendo “No tengo información sobre eso”).  
- Entender que la calidad de las respuestas **depende completamente de la base de conocimiento cargada**.

---

### 📝 Ejemplo

Si nuestros documentos hablan sobre **Argentina**, y el usuario pregunta:

> “¿Quién es el presidente de Japón?”

👉 El sistema no encontrará información relevante, porque esa pregunta **no está cubierta en el corpus**.  
Por lo tanto, **no debería poder responder correctamente**.

---

### ✨ En pocas palabras

> **El sistema solo puede generar respuestas basadas en la información que realmente tiene en sus documentos.**  
> Si algo no está en la base, simplemente **no podrá (ni deberá) desplegar esa información**.


In [ ]:
# armando una lista de preguntas para probar nuestro sistema RAG

query_list = [
    "¿Cual es la capital de Argentina?",
    "Que es un choripan?"]

for query in query_list:
    rag(query)
    
# probamos que en uno de los casos no tenemos la informacion en nuestros documentos
# y el modelo responde "No lo se" como le indicamos en el prompt

 ¡Hola, amigos! Preguntan cuál es la capital de Argentina, así que ¡allá vamos! Según el primer documento, Buenos Aires es la capital y ciudad más grande de Argentina. ¡Eso significa que la capital de Argentina es Buenos Aires, mis queridos turistas!

 Lo siento, pero ninguno de los documentos proporcionados contiene información sobre qué es un "choripan". El primer documento habla sobre la salsa chimichurri, el segundo sobre el glaciar Perito Moreno, y el tercero sobre el idioma oficial de Argentina. El término "choripan" no está mencionado en ninguno de ellos. "Choripan" es un popular plato callejero argentino que consiste en una salchicha (chorizo) cocida y servida en un pan (pan).